# LeetCode 3534 — Path Existence Queries in a Graph II bằng Python

- Link: https://leetcode.com/problems/path-existence-queries-in-a-graph-ii/
- Độ khó: **Hard**
- Chủ đề: **Sắp xếp**, **hai con trỏ** (bước nhảy xa nhất), **tham lam nhảy xa nhất**, **binary lifting trên hàm "nhảy xa nhất"**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def pathExistenceQueries(self, n: int, nums: List[int], maxDiff: int, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra đồ thị "nối `i`–`j` khi `|nums[i] − nums[j]| ≤ maxDiff`" sau khi **sắp xếp theo giá trị** trở thành đồ thị khoảng: hàng xóm của một đỉnh là một **đoạn liên tiếp**.
2. Chứng minh đường đi ngắn nhất từ vị trí `a` tới `b` (`a < b` trong thứ tự sắp) đạt được bằng **tham lam nhảy xa nhất** về phía `b`.
3. Tính `far[i]` (chỉ số xa nhất nối được từ `i`) bằng **hai con trỏ** `O(n)`.
4. Trả lời mỗi truy vấn trong `O(log n)` bằng **binary lifting** trên hàm `far`.
5. Kiểm chứng bằng BFS trên đồ thị dựng tường minh với dữ liệu nhỏ.

## 2. Đề bài & ví dụ

Đồ thị `n` đỉnh `0..n−1`; giữa `i` và `j` có cạnh (vô hướng, không trọng số) nếu `|nums[i] − nums[j]| ≤ maxDiff`. Với mỗi truy vấn `[u, v]` trả về **khoảng cách ngắn nhất** (số cạnh) giữa `u` và `v`, hoặc `−1` nếu không có đường.

**Ví dụ 1:** `n=5, nums=[1,8,3,4,2], maxDiff=3, queries=[[0,3],[2,4]]` → `[1,1]`.

**Ví dụ 2:** `n=5, nums=[5,3,1,9,10], maxDiff=2, queries=[[0,1],[0,2],[2,3],[4,3]]` → `[1,2,-1,1]`.

**Ví dụ 3:** `n=3, nums=[3,6,1], maxDiff=1, queries=[[0,0],[0,1],[1,2]]` → `[0,-1,-1]`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `0 ≤ nums[i], maxDiff ≤ 10^5`, `1 ≤ len(queries) ≤ 10^5`.

In [1]:
from typing import List
from collections import deque
examples = [
    (5, [1, 8, 3, 4, 2], 3, [[0, 3], [2, 4]], [1, 1]),
    (5, [5, 3, 1, 9, 10], 2, [[0, 1], [0, 2], [2, 3], [4, 3]], [1, 2, -1, 1]),
    (3, [3, 6, 1], 1, [[0, 0], [0, 1], [1, 2]], [0, -1, -1]),
]
for n, nums, md, q, exp in examples:
    print(n, nums, md, q, '-> kỳ vọng', exp)

5 [1, 8, 3, 4, 2] 3 [[0, 3], [2, 4]] -> kỳ vọng [1, 1]
5 [5, 3, 1, 9, 10] 2 [[0, 1], [0, 2], [2, 3], [4, 3]] -> kỳ vọng [1, 2, -1, 1]
3 [3, 6, 1] 1 [[0, 0], [0, 1], [1, 2]] -> kỳ vọng [0, -1, -1]


## 3. Kiến thức nền

### 3.1. Sắp xếp chỉ số theo giá trị và "hạng"

`order = sorted(range(n), key=nums.__getitem__)`; `rank[order[r]] = r`. Sau đó `vals[r] = nums[order[r]]` là dãy không giảm.

In [2]:
nums = [5, 3, 1, 9, 10]
order = sorted(range(len(nums)), key=nums.__getitem__)
rank = [0] * len(nums)
for r, i in enumerate(order):
    rank[i] = r
print('order =', order, '| rank =', rank, '| vals =', [nums[i] for i in order])

order = [2, 1, 0, 3, 4] | rank = [2, 1, 0, 3, 4] | vals = [1, 3, 5, 9, 10]


### 3.2. Hai con trỏ cho `far[i]`

`far[i]` = chỉ số lớn nhất `j` với `vals[j] − vals[i] ≤ maxDiff`. Khi `i` tăng, `far[i]` không giảm ⇒ con trỏ `j` chỉ tiến ⇒ `O(n)`.

In [3]:
def far_array(vals, maxDiff):
    n = len(vals)
    far, j = [0] * n, 0
    for i in range(n):
        j = max(j, i)
        while j + 1 < n and vals[j + 1] - vals[i] <= maxDiff:
            j += 1
        far[i] = j
    return far

vals = sorted([5, 3, 1, 9, 10])
print('vals =', vals, '| far =', far_array(vals, 2))

vals = [1, 3, 5, 9, 10] | far = [1, 2, 2, 4, 4]


### 3.3. Binary lifting trên một hàm

Cho hàm `f: {0..n−1} → {0..n−1}`, `up[0] = f`, `up[j][x] = up[j−1][up[j−1][x]]` là `f` áp dụng `2^j` lần. Tìm số lần áp dụng nhỏ nhất để đạt điều kiện đơn điệu: nhảy `2^j` giảm dần khi **chưa** đạt điều kiện.

In [4]:
f = far_array(sorted([1, 2, 3, 5, 7, 9, 12]), 2)
up = [f]
for _ in range(3):
    up.append([up[-1][up[-1][x]] for x in range(len(f))])
print('f =', f)
for j, row in enumerate(up):
    print(f"f^(2^{j}) =", row)

f = [2, 2, 3, 4, 5, 5, 6]
f^(2^0) = [2, 2, 3, 4, 5, 5, 6]
f^(2^1) = [3, 3, 4, 5, 5, 5, 6]
f^(2^2) = [5, 5, 5, 5, 5, 5, 6]
f^(2^3) = [5, 5, 5, 5, 5, 5, 6]


### 3.4. BFS (để kiểm chứng)

BFS trên đồ thị không trọng số cho khoảng cách ngắn nhất; dùng làm oracle trên đồ thị dựng tường minh `O(n^2)` cạnh.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Sau khi sắp xếp, mỗi đỉnh nối với **mọi** đỉnh có giá trị trong `[v − maxDiff, v + maxDiff]` — một đoạn liên tiếp các vị trí. Muốn đi từ vị trí `a` sang `b > a` nhanh nhất: mỗi bước nhảy tới vị trí **xa nhất** `far[x]` (về phía `b`).

Tính tay ví dụ 2: `vals = [1, 3, 5, 9, 10]` (đỉnh gốc `[2, 1, 0, 3, 4]`), `maxDiff = 2`, `far = [1, 2, 2, 4, 4]`.

- `[0, 2]` (đỉnh 0 có giá trị 5 → vị trí 2; đỉnh 2 giá trị 1 → vị trí 0): `a = 0, b = 2`. Từ 0 nhảy tới `far[0] = 1` (chưa tới 2), từ 1 nhảy tới `far[1] = 2` ⇒ **2 bước**.
- `[2, 3]`: `a = 0, b = 3`. `0 → 1 → 2 → 2` kẹt (far[2] = 2 < 3) ⇒ `−1`.

In [5]:
def greedy_steps(far, a, b):
    steps, x = 0, a
    while x < b:
        if far[x] == x:
            return -1                     # không tiến được nữa
        x = far[x]
        steps += 1
    return steps

far = [1, 2, 2, 4, 4]
print(greedy_steps(far, 0, 2), greedy_steps(far, 0, 3), greedy_steps(far, 3, 4))

2 -1 1


## 5. Cách 1 — Brute Force: dựng đồ thị tường minh + BFS mỗi truy vấn

In [6]:
def path_queries_brute(n, nums, maxDiff, queries):
    adj = [[j for j in range(n) if j != i and abs(nums[i] - nums[j]) <= maxDiff] for i in range(n)]
    out = []
    for u, v in queries:
        dist = {u: 0}
        dq = deque([u])
        while dq:
            x = dq.popleft()
            for y in adj[x]:
                if y not in dist:
                    dist[y] = dist[x] + 1
                    dq.append(y)
        out.append(dist.get(v, -1))
    return out

for n, nums, md, q, exp in examples:
    print(path_queries_brute(n, nums, md, q), '| kỳ vọng', exp)

[1, 1] | kỳ vọng [1, 1]
[1, 2, -1, 1] | kỳ vọng [1, 2, -1, 1]
[0, -1, -1] | kỳ vọng [0, -1, -1]


**Độ phức tạp:** dựng đồ thị `O(n^2)`, mỗi truy vấn BFS `O(n^2)` ⇒ không dùng được với `n = 10^5`.

**Nút thắt:** số cạnh có thể `Θ(n^2)` (mọi giá trị gần nhau). Nhưng cấu trúc "đoạn liên tiếp sau khi sắp xếp" cho phép đi **tham lam**, không cần cạnh tường minh.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề tham lam.** Trong thứ tự sắp xếp, gọi `g_t(a)` = vị trí xa nhất (về phía phải) đạt được từ `a` sau **tối đa** `t` bước. Khi đó `g_{t+1}(a) = far[g_t(a)]`.

*Lý do:* sau `t` bước, tập vị trí đạt được (về phía phải của `a`) là đoạn `[a, g_t(a)]` (đồ thị khoảng: nếu đạt được `y` thì đạt được mọi vị trí giữa `a` và `y` trong cùng số bước hoặc ít hơn). Bước tiếp theo đi xa nhất từ vị trí có giá trị lớn nhất, tức `g_t(a)`, và vì `far` không giảm nên `far[g_t(a)]` là xa nhất.

⇒ khoảng cách `(a, b)` với `a < b` = số `t` nhỏ nhất sao cho `far^t(a) ≥ b`; nếu `far^t(a)` dừng lại (`far[x] = x`) trước `b` thì `−1`.

**Tăng tốc:** binary lifting `up[j] = far^{2^j}`. Với truy vấn: nhảy `2^j` giảm dần **khi đích vẫn `< b`**, cộng dồn số bước; cuối cùng thêm 1 bước nếu `far[x] ≥ b`.

Trường hợp đặc biệt: `u = v` ⇒ 0. Hai đỉnh khác nhau có cùng giá trị ⇒ `far[a] ≥ b` ⇒ 1.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS trên thứ tự sắp xếp với "tập chưa thăm" dạng đoạn

Hàng xóm của vị trí `x` là đoạn `[lo[x], far[x]]`. Duy trì con trỏ "vị trí chưa thăm kế tiếp" (DSU nhảy) để mỗi vị trí chỉ bị thêm vào hàng đợi một lần ⇒ BFS `O(n α(n))` mỗi truy vấn — vẫn `O(n q)` tổng.

In [7]:
def path_queries_bfs_sorted(n, nums, maxDiff, queries):
    order = sorted(range(n), key=nums.__getitem__)
    rank = [0] * n
    for r, i in enumerate(order):
        rank[i] = r
    vals = [nums[i] for i in order]
    far = far_array(vals, maxDiff)
    lo, j = [0] * n, 0
    for i in range(n):
        while vals[i] - vals[j] > maxDiff:
            j += 1
        lo[i] = j
    out = []
    for u, v in queries:
        a, b = rank[u], rank[v]
        nxt = list(range(n + 1))          # nxt[x] = vị trí chưa thăm nhỏ nhất >= x

        def find(x):
            while nxt[x] != x:
                nxt[x] = nxt[nxt[x]]
                x = nxt[x]
            return x

        dist = {a: 0}
        nxt[a] = a + 1
        dq = deque([a])
        while dq:
            x = dq.popleft()
            y = find(lo[x])
            while y <= far[x]:
                dist[y] = dist[x] + 1
                nxt[y] = y + 1
                dq.append(y)
                y = find(y)
        out.append(dist.get(b, -1))
    return out

for n, nums, md, q, exp in examples:
    print(path_queries_bfs_sorted(n, nums, md, q), '| kỳ vọng', exp)

[1, 1] | kỳ vọng [1, 1]
[1, 2, -1, 1] | kỳ vọng [1, 2, -1, 1]
[0, -1, -1] | kỳ vọng [0, -1, -1]


### 7.2. Cách 3 — Tham lam nhảy xa nhất, mô phỏng từng bước

Dựa vào bổ đề: lặp `x = far[x]` cho tới khi `x ≥ b` hoặc kẹt. `O(khoảng cách)` mỗi truy vấn — xấu nhất `O(n)` (ví dụ dãy giá trị cách đều đúng `maxDiff`).

In [8]:
def path_queries_greedy(n, nums, maxDiff, queries):
    order = sorted(range(n), key=nums.__getitem__)
    rank = [0] * n
    for r, i in enumerate(order):
        rank[i] = r
    far = far_array([nums[i] for i in order], maxDiff)
    out = []
    for u, v in queries:
        if u == v:
            out.append(0)
            continue
        a, b = sorted((rank[u], rank[v]))
        out.append(greedy_steps(far, a, b))
    return out

for n, nums, md, q, exp in examples:
    print(path_queries_greedy(n, nums, md, q), '| kỳ vọng', exp)

[1, 1] | kỳ vọng [1, 1]
[1, 2, -1, 1] | kỳ vọng [1, 2, -1, 1]
[0, -1, -1] | kỳ vọng [0, -1, -1]


### 7.3. Cách 4 — Binary lifting (lời giải chính)

In [9]:
def path_queries_lifting(n, nums, maxDiff, queries):
    order = sorted(range(n), key=nums.__getitem__)
    rank = [0] * n
    for r, i in enumerate(order):
        rank[i] = r
    far = far_array([nums[i] for i in order], maxDiff)
    LOG = max(1, n.bit_length())
    up = [far]
    for _ in range(1, LOG):
        pr = up[-1]
        up.append([pr[pr[x]] for x in range(n)])
    out = []
    for u, v in queries:
        if u == v:
            out.append(0)
            continue
        a, b = sorted((rank[u], rank[v]))
        x, steps = a, 0
        for j in range(LOG - 1, -1, -1):
            y = up[j][x]
            if y < b:
                x, steps = y, steps + (1 << j)
        out.append(steps + 1 if far[x] >= b else -1)
    return out

for n, nums, md, q, exp in examples:
    print(path_queries_lifting(n, nums, md, q), '| kỳ vọng', exp)

[1, 1] | kỳ vọng [1, 1]
[1, 2, -1, 1] | kỳ vọng [1, 2, -1, 1]
[0, -1, -1] | kỳ vọng [0, -1, -1]


Khi kẹt (`far[x] = x < b`), mọi `up[j][x] = x < b` nên vòng lặp cộng đủ `2^LOG − 1` bước nhưng kết quả vẫn trả `−1` vì `far[x] < b` — đúng.

**Độ phức tạp:** sắp xếp `O(n log n)`, bảng `O(n log n)`, mỗi truy vấn `O(log n)`.

## 8. TRACE từng bước

Trace cách 4 trên một dãy dài hơn: `nums = [0, 2, 4, 6, 8, 10, 12, 14, 30]`, `maxDiff = 2` (mỗi bước chỉ tiến 1 vị trí, tới 30 thì kẹt).

In [10]:
def trace(nums, maxDiff, queries):
    n = len(nums)
    order = sorted(range(n), key=nums.__getitem__)
    rank = [0] * n
    for r, i in enumerate(order):
        rank[i] = r
    vals = [nums[i] for i in order]
    far = far_array(vals, maxDiff)
    LOG = max(1, n.bit_length())
    up = [far]
    for _ in range(1, LOG):
        up.append([up[-1][up[-1][x]] for x in range(n)])
    print('vals =', vals, '| far =', far)
    for u, v in queries:
        a, b = sorted((rank[u], rank[v]))
        x, steps, log = a, 0, []
        for j in range(LOG - 1, -1, -1):
            y = up[j][x]
            if y < b:
                log.append(f"nhảy 2^{j}: {x}->{y}")
                x, steps = y, steps + (1 << j)
        ans = steps + 1 if far[x] >= b else -1
        print(f"[{u},{v}] (vị trí {a} -> {b}): {', '.join(log) or 'không nhảy'}; far[{x}] = {far[x]} -> đáp án {ans}")

trace([0, 2, 4, 6, 8, 10, 12, 14, 30], 2, [[0, 7], [0, 5], [3, 4], [0, 8]])

vals = [0, 2, 4, 6, 8, 10, 12, 14, 30] | far = [1, 2, 3, 4, 5, 6, 7, 7, 8]
[0,7] (vị trí 0 -> 7): nhảy 2^2: 0->4, nhảy 2^1: 4->6; far[6] = 7 -> đáp án 7
[0,5] (vị trí 0 -> 5): nhảy 2^2: 0->4; far[4] = 5 -> đáp án 5
[3,4] (vị trí 3 -> 4): không nhảy; far[3] = 4 -> đáp án 1
[0,8] (vị trí 0 -> 8): nhảy 2^3: 0->7, nhảy 2^2: 7->7, nhảy 2^1: 7->7, nhảy 2^0: 7->7; far[7] = 7 -> đáp án -1


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def pathExistenceQueries(self, n: int, nums: List[int], maxDiff: int, queries: List[List[int]]) -> List[int]:
        order = sorted(range(n), key=nums.__getitem__)
        rank = [0] * n
        for r, i in enumerate(order):
            rank[i] = r
        vals = [nums[i] for i in order]
        far = [0] * n
        j = 0
        for i in range(n):
            if j < i:
                j = i
            while j + 1 < n and vals[j + 1] - vals[i] <= maxDiff:
                j += 1
            far[i] = j
        LOG = max(1, n.bit_length())
        up = [far]
        for _ in range(1, LOG):
            pr = up[-1]
            up.append([pr[pr[x]] for x in range(n)])

        res = []
        for u, v in queries:
            if u == v:
                res.append(0)
                continue
            a, b = rank[u], rank[v]
            if a > b:
                a, b = b, a
            if far[a] >= b:
                res.append(1)
                continue
            x, steps = a, 0
            for jj in range(LOG - 1, -1, -1):
                y = up[jj][x]
                if y < b:
                    x = y
                    steps += 1 << jj
            res.append(steps + 1 if far[x] >= b else -1)
        return res

for n, nums, md, q, exp in examples:
    got = Solution().pathExistenceQueries(n, nums, md, q)
    print(got, 'OK' if got == exp else 'SAI')

[1, 1] OK
[1, 2, -1, 1] OK
[0, -1, -1] OK


## 10. Vì sao thuật toán đúng

1. **Đồ thị khoảng:** sau khi sắp xếp, vị trí `x` nối với đúng các vị trí trong `[lo[x], far[x]]` (vì `vals` không giảm). `far` và `lo` đều không giảm theo `x`.
2. **Đi về một phía là đủ:** với `a < b`, một đường đi ngắn nhất có thể chọn sao cho dãy vị trí **tăng dần** — nếu đường đi lùi về trái rồi lại sang phải, vì `far` không giảm, thay bước lùi bằng bước tiến xa nhất không làm tệ hơn.
3. **Tham lam xa nhất tối ưu:** bằng quy nạp, sau `t` bước, `far^t(a)` là vị trí xa nhất đạt được trong `≤ t` bước, và mọi vị trí trong `[a, far^t(a)]` cũng đạt được trong `≤ t` bước. Vậy `b` đạt được trong `t` bước ⇔ `far^t(a) ≥ b`.
4. **Binary lifting:** hàm `t ↦ far^t(a)` không giảm; tìm `t` nhỏ nhất với `far^t(a) ≥ b` bằng nhảy `2^j` giảm dần khi đích `< b`, rồi cộng 1.
5. **Không có đường:** nếu `far[x] = x < b` thì `far^t(a)` dừng lại mãi ở `x` ⇒ `−1`. Hai đỉnh cùng thành phần ⇔ không có "khoảng trống" `> maxDiff` giữa chúng trong `vals`.

## 11. Phân tích Time / Space Complexity

Gọi `n` số đỉnh, `q` số truy vấn.

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Đồ thị tường minh + BFS | `O(n^2)` | `O(n^2)` | `O(n^2)` |
| 2. BFS trên thứ tự sắp xếp | `O(n log n)` | `O(n α(n))` | `O(n)` |
| 3. Tham lam từng bước | `O(n log n)` | `O(khoảng cách) ≤ O(n)` | `O(n)` |
| 4. Binary lifting | `O(n log n)` | `O(log n)` | `O(n log n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {
    'bfs_sorted': path_queries_bfs_sorted,
    'greedy': path_queries_greedy,
    'lifting': path_queries_lifting,
    'submit': lambda *a: Solution().pathExistenceQueries(*a),
}
edge = [
    (1, [7], 0, [[0, 0]], [0]),
    (2, [4, 4], 0, [[0, 1], [1, 0]], [1, 1]),          # cùng giá trị, maxDiff = 0
    (2, [4, 5], 0, [[0, 1]], [-1]),
    (4, [0, 3, 6, 9], 3, [[0, 3], [3, 0], [1, 2]], [3, 3, 1]),
    (4, [9, 0, 6, 3], 3, [[1, 0], [0, 1]], [3, 3]),     # thứ tự đầu vào bị xáo
]
for n, nums, md, q, exp in examples + edge:
    for name, f in impls.items():
        got = f(n, nums, md, q)
        assert got == exp, (name, n, nums, md, q, got, exp)
    assert path_queries_brute(n, nums, md, q) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với BFS tường minh

In [13]:
import random
random.seed(3534)
cnt = 0
for _ in range(500):
    n = random.randint(1, 14)
    nums = [random.randint(0, 25) for _ in range(n)]
    md = random.randint(0, 6)
    q = [[random.randrange(n), random.randrange(n)] for _ in range(10)]
    exp = path_queries_brute(n, nums, md, q)
    for name, f in impls.items():
        assert f(n, nums, md, q) == exp, (name, n, nums, md, q)
    cnt += 1
print('Khớp BFS tường minh trên', cnt, 'bộ test ngẫu nhiên.')

Khớp BFS tường minh trên 500 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 10^5`)

In [14]:
import time
random.seed(5)
n = 10**5
qs = [[random.randrange(n), random.randrange(n)] for _ in range(n)]
cases = {
    'ngẫu nhiên, maxDiff=3': ([random.randint(0, 10**5) for _ in range(n)], 3),
    'cách đều (đường dài)': ([2 * i for i in range(n)], 2),
    'maxDiff lớn': ([random.randint(0, 10**5) for _ in range(n)], 10**5),
}
for name, (nums, md) in cases.items():
    t = time.time(); a = Solution().pathExistenceQueries(n, nums, md, qs); t1 = time.time() - t
    b = path_queries_greedy(n, nums, md, qs[:200])
    assert a[:200] == b
    print(f"{name:>22}: {t1:.2f}s | 5 đáp án đầu {a[:5]}")

 ngẫu nhiên, maxDiff=3: 0.40s | 5 đáp án đầu [-1, -1, -1, -1, -1]


  cách đều (đường dài): 0.43s | 5 đáp án đầu [48163, 50234, 6481, 15990, 57229]
           maxDiff lớn: 0.14s | 5 đáp án đầu [1, 1, 1, 1, 1]


Ca "cách đều" là trường hợp xấu cho cách 3 (khoảng cách tới `~10^5` bước mỗi truy vấn) nhưng binary lifting vẫn `O(log n)`.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra cùng thành phần liên thông, trả 1 cho mọi cặp nối được

In [15]:
def wrong_connectivity_only(n, nums, maxDiff, queries):
    order = sorted(range(n), key=nums.__getitem__)
    comp, c = [0] * n, 0
    for r in range(1, n):
        if nums[order[r]] - nums[order[r - 1]] > maxDiff:
            c += 1
        comp[order[r]] = c
    return [0 if u == v else (1 if comp[u] == comp[v] else -1) for u, v in queries]   # SAI

n, nums, md, q, exp = examples[1]
print('sai :', wrong_connectivity_only(n, nums, md, q), '| đúng:', Solution().pathExistenceQueries(n, nums, md, q))

sai : [1, 1, -1, 1] | đúng: [1, 2, -1, 1]


**Vì sao sai:** đề hỏi **khoảng cách ngắn nhất**, không chỉ có đường hay không.

### Lỗi 2 — Quên ánh xạ đỉnh gốc sang vị trí trong mảng đã sắp

In [16]:
def wrong_no_rank(n, nums, maxDiff, queries):
    vals = sorted(nums)
    far = far_array(vals, maxDiff)
    out = []
    for u, v in queries:
        a, b = sorted((u, v))                   # SAI: dùng chỉ số gốc như vị trí sắp xếp
        out.append(0 if u == v else greedy_steps(far, a, b))
    return out

case = (5, [1, 8, 3, 4, 2], 3, [[0, 3], [2, 4], [0, 1], [1, 3]])     # ví dụ 1 + vài truy vấn
print('sai :', wrong_no_rank(*case), '| đúng:', Solution().pathExistenceQueries(*case))

sai : [1, -1, 1, 1] | đúng: [1, 1, -1, -1]


**Vì sao sai:** sau khi sắp xếp, đỉnh `u` nằm ở vị trí `rank[u]`, không phải `u`.

### Lỗi 3 — Trả 0 cho hai đỉnh khác nhau có cùng giá trị

In [17]:
def wrong_same_value(n, nums, maxDiff, queries):
    res = Solution().pathExistenceQueries(n, nums, maxDiff, queries)
    return [0 if nums[u] == nums[v] else r for (u, v), r in zip(queries, res)]     # SAI

print('sai :', wrong_same_value(2, [4, 4], 0, [[0, 1]]), '| đúng:', Solution().pathExistenceQueries(2, [4, 4], 0, [[0, 1]]))

sai : [0] | đúng: [1]


**Vì sao sai:** khoảng cách tính theo **đỉnh**; hai đỉnh khác nhau cần ít nhất một cạnh.

### Lỗi 4 — Binary lifting nhảy khi `y ≤ b` (thay vì `y < b`) rồi vẫn cộng 1

In [18]:
def wrong_le(n, nums, maxDiff, queries):
    order = sorted(range(n), key=nums.__getitem__)
    rank = [0] * n
    for r, i in enumerate(order):
        rank[i] = r
    far = far_array([nums[i] for i in order], maxDiff)
    LOG = max(1, n.bit_length())
    up = [far]
    for _ in range(1, LOG):
        up.append([up[-1][up[-1][x]] for x in range(n)])
    out = []
    for u, v in queries:
        if u == v:
            out.append(0); continue
        a, b = sorted((rank[u], rank[v]))
        x, s = a, 0
        for j in range(LOG - 1, -1, -1):
            if up[j][x] <= b:                    # SAI
                x, s = up[j][x], s + (1 << j)
        out.append(s + 1 if far[x] >= b else -1)
    return out

case = (4, [0, 3, 6, 9], 3, [[0, 3]])
print('sai :', wrong_le(*case), '| đúng:', Solution().pathExistenceQueries(*case))

sai : [8] | đúng: [3]


**Vì sao sai:** với `≤`, vòng nhảy có thể đã **tới** `b` rồi mà vẫn cộng thêm 1 bước (và khi kẹt đúng tại `b` còn cộng bước thừa). Điều kiện chuẩn: nhảy khi đích **chưa tới** `b`, cuối cùng cộng đúng 1.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Cạnh định nghĩa bởi "`|a − b| ≤ d`" ⇒ **sắp xếp**; hàng xóm thành đoạn liên tiếp (đồ thị khoảng).
- Đường ngắn nhất trên đồ thị khoảng ⇒ **tham lam nhảy xa nhất** (giống LC 45 *Jump Game II*).
- Nhiều truy vấn "bao nhiêu bước nhảy" ⇒ **binary lifting** trên hàm bước nhảy.
- `far` tính bằng **hai con trỏ** vì đơn điệu.

## 15. Biến thể của bài

1. Bản I (LC 3532, mảng đã sắp, chỉ hỏi có đường hay không) ⇒ đánh số thành phần bằng khoảng trống `> maxDiff`.
2. Cạnh có trọng số `|nums[i] − nums[j]|` ⇒ đường ngắn nhất theo trọng số = `|nums[u] − nums[v]|` nếu liên thông (đi tuần tự qua các giá trị giữa).
3. Truy vấn **số bước tối đa** `t`: hỏi đỉnh xa nhất đạt được trong `t` bước ⇒ `far^t(a)` bằng binary lifting.
4. Bài liên quan: LC 45 *Jump Game II*, LC 1696 *Jump Game VI*, LC 2836 *Maximize Value of Function in a Ball Passing Game* (binary lifting trên hàm).

In [19]:
# Biến thể 3: vị trí xa nhất (về phải) đạt được trong t bước
def reach_in_t(nums, maxDiff, u, t):
    n = len(nums)
    order = sorted(range(n), key=nums.__getitem__)
    rank = {i: r for r, i in enumerate(order)}
    far = far_array([nums[i] for i in order], maxDiff)
    LOG = max(1, max(t, 1).bit_length())
    up = [far]
    for _ in range(1, LOG):
        up.append([up[-1][up[-1][x]] for x in range(n)])
    x, j = rank[u], 0
    while t:
        if t & 1:
            x = up[j][x]
        t >>= 1
        j += 1
    return nums[order[x]]

print('giá trị lớn nhất đạt được từ đỉnh 0 trong 3 bước:', reach_in_t([0, 2, 4, 6, 8, 10, 12, 14, 30], 2, 0, 3))

giá trị lớn nhất đạt được từ đỉnh 0 trong 3 bước: 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Đồ thị tường minh + BFS | `O(n^2)` | `O(n^2)` | Oracle |
| 2 | BFS trên thứ tự sắp xếp | `O(n α)` | `O(n)` | Khi cần khoảng cách tới **mọi** đỉnh |
| 3 | Tham lam từng bước | `O(khoảng cách)` | `O(n)` | Khoảng cách nhỏ |
| 4 | Binary lifting | `O(log n)` | `O(n log n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n=4, nums=[1, 10, 4, 7], maxDiff=3`, truy vấn `[[0, 1], [2, 3], [1, 1]]`.

**Bài 2 (tính tay):** `nums=[0, 5, 10, 11, 20]`, `maxDiff=5`, truy vấn `[[0, 4], [0, 3]]`.

**Bài 3 (điền TODO):** viết `components(nums, maxDiff)` trả về nhãn thành phần của mỗi đỉnh (biến thể 1).

**Bài 4 (tự viết lại):** biến thể 2 — khoảng cách **có trọng số** `|nums[i] − nums[j]|`; kiểm chứng bằng Dijkstra.

**Bài 5 (mini challenge):** in ra **một đường đi ngắn nhất** cụ thể (dãy đỉnh) cho mỗi truy vấn bằng tham lam, và kiểm tra mọi cạnh hợp lệ.

In [20]:
def components_todo(nums, maxDiff):
    # TODO
    raise NotImplementedError

def weighted_dist_todo(nums, maxDiff, queries):
    # TODO
    raise NotImplementedError

def shortest_path_nodes_todo(nums, maxDiff, u, v):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `vals = [1, 4, 7, 10]`, mỗi bước tiến 1. (2) `0 → 1`: từ giá trị 1 tới 10: 3 bước; `2 → 3`: 4 với 7: 1 bước. (3) `[3, 1, 0]`.

**Bài 2.** (1) `vals = [0,5,10,11,20]`, `far = [1,2,3,3,4]`. (2) `0 → 4`: kẹt ở vị trí 3 (11 → 20 > 5) ⇒ `−1`. (3) `0 → 3`: `0 → 1 → 2 → 3` ⇒ 3.

**Bài 3.** (1) Sắp xếp. (2) Tăng nhãn khi gặp khoảng trống `> maxDiff`. (3) Gán lại theo chỉ số gốc.

**Bài 4.** (1) Nếu cùng thành phần: `|nums[u] − nums[v]|`. (2) Vì đi qua các giá trị trung gian theo thứ tự tăng, tổng trọng số "trượt" đúng bằng hiệu. (3) Dijkstra trên đồ thị nhỏ để kiểm.

**Bài 5.** (1) Đi từ vị trí nhỏ hơn, mỗi bước tới `far[x]` nhưng không vượt `b` (nếu `far[x] ≥ b` thì nhảy thẳng tới `b`). (2) Đổi vị trí về đỉnh gốc bằng `order`. (3) Kiểm tra `|nums| ≤ maxDiff` cho mọi cạnh và số cạnh = đáp án.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
a1 = Solution().pathExistenceQueries(4, [1, 10, 4, 7], 3, [[0, 1], [2, 3], [1, 1]])
a2 = Solution().pathExistenceQueries(5, [0, 5, 10, 11, 20], 5, [[0, 4], [0, 3]])
assert a1 == [3, 1, 0] == path_queries_brute(4, [1, 10, 4, 7], 3, [[0, 1], [2, 3], [1, 1]])
assert a2 == [-1, 3] == path_queries_brute(5, [0, 5, 10, 11, 20], 5, [[0, 4], [0, 3]])
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: [3, 1, 0] | Bài 2: [-1, 3]


In [22]:
# Bài 3, 4
import heapq

def components(nums, maxDiff):
    n = len(nums)
    order = sorted(range(n), key=nums.__getitem__)
    comp, c = [0] * n, 0
    for r in range(1, n):
        if nums[order[r]] - nums[order[r - 1]] > maxDiff:
            c += 1
        comp[order[r]] = c
    return comp

def weighted_dist(nums, maxDiff, queries):
    comp = components(nums, maxDiff)
    return [abs(nums[u] - nums[v]) if comp[u] == comp[v] else -1 for u, v in queries]

def dijkstra_check(nums, maxDiff, queries):
    n = len(nums)
    out = []
    for u, v in queries:
        dist = [float('inf')] * n
        dist[u] = 0
        pq = [(0, u)]
        while pq:
            d, x = heapq.heappop(pq)
            if d > dist[x]:
                continue
            for y in range(n):
                w = abs(nums[x] - nums[y])
                if y != x and w <= maxDiff and d + w < dist[y]:
                    dist[y] = d + w
                    heapq.heappush(pq, (d + w, y))
        out.append(dist[v] if dist[v] < float('inf') else -1)
    return out

random.seed(6)
for _ in range(200):
    n = random.randint(1, 10)
    nums = [random.randint(0, 20) for _ in range(n)]
    md = random.randint(0, 5)
    q = [[random.randrange(n), random.randrange(n)] for _ in range(6)]
    assert weighted_dist(nums, md, q) == dijkstra_check(nums, md, q)
    # thành phần: cùng nhãn <=> có đường
    assert [comp_eq for comp_eq in [components(nums, md)[u] == components(nums, md)[v] for u, v in q]] == \
           [d != -1 for d in path_queries_brute(n, nums, md, q)]
print('Bài 3 & 4: khớp BFS / Dijkstra')

Bài 3 & 4: khớp BFS / Dijkstra


In [23]:
# Bài 5: một đường đi ngắn nhất cụ thể
def shortest_path_nodes(nums, maxDiff, u, v):
    n = len(nums)
    if u == v:
        return [u]
    order = sorted(range(n), key=nums.__getitem__)
    rank = [0] * n
    for r, i in enumerate(order):
        rank[i] = r
    far = far_array([nums[i] for i in order], maxDiff)
    a, b = rank[u], rank[v]
    flip = a > b
    if flip:
        a, b = b, a
    path, x = [a], a
    while x < b:
        if far[x] == x:
            return None
        x = min(far[x], b)
        path.append(x)
    nodes = [order[p] for p in path]
    return nodes[::-1] if flip else nodes

random.seed(7)
for _ in range(300):
    n = random.randint(1, 12)
    nums = [random.randint(0, 20) for _ in range(n)]
    md = random.randint(0, 5)
    u, v = random.randrange(n), random.randrange(n)
    d = path_queries_brute(n, nums, md, [[u, v]])[0]
    p = shortest_path_nodes(nums, md, u, v)
    if d == -1:
        assert p is None
    else:
        assert p[0] == u and p[-1] == v and len(p) - 1 == d
        assert all(abs(nums[p[i]] - nums[p[i + 1]]) <= md for i in range(len(p) - 1))
print('Bài 5: ví dụ', shortest_path_nodes([5, 3, 1, 9, 10], 2, 0, 2))

Bài 5: ví dụ [0, 1, 2]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
order = sắp chỉ số theo nums; rank[order[r]] = r; vals = nums theo order
far[i] = j lớn nhất với vals[j] - vals[i] <= maxDiff     (hai con trỏ)
up[0] = far; up[j][x] = up[j-1][up[j-1][x]]
truy vấn: u == v -> 0; a, b = sorted(rank[u], rank[v])
    x = a; steps = 0; với j giảm dần: nếu up[j][x] < b: x = up[j][x]; steps += 2^j
    đáp án = steps + 1 nếu far[x] >= b, ngược lại -1
```

**Checklist:**

- [ ] Ánh xạ đỉnh → vị trí sau sắp xếp (`rank`)?
- [ ] `u = v` ⇒ 0; cùng giá trị khác đỉnh ⇒ 1?
- [ ] Nhảy khi đích **< b**, cộng 1 ở cuối?
- [ ] Trường hợp kẹt (`far[x] = x`) trả `−1`?
- [ ] `far` bằng hai con trỏ `O(n)`?